In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

ZIP_NAME = 'h5_data'
METADATA_NAME = 'ground_truth.csv'

# Set up Google Drive paths
DRIVE_PATH = '/content/drive/My Drive/Software Projects/skin_lesion_classifier/'

DATA_PATH = os.path.join(DRIVE_PATH, 'data')
METADATA_PATH = os.path.join(os.path.join(DATA_PATH, 'metadata'), METADATA_NAME)

SCRIPTS_PATH = os.path.join(DRIVE_PATH, 'scripts')
MODELS_PATH = os.path.join(DRIVE_PATH, 'models')
RESULTS_PATH = os.path.join(DRIVE_PATH, 'results')

ZIP_PATH = os.path.join(DRIVE_PATH, f'{ZIP_NAME}.zip')

In [ ]:
import shutil

# Copy the h5 zip from Google Drive to the local environment
shutil.copy2(ZIP_PATH, '/content/')
print("zip copied successfully.")

zip copied successfully.


In [ ]:
import zipfile

print("unzipping...")

def unzip_file(zip_file_path, extract_dir):
  with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

unzip_file(f'/content/{ZIP_NAME}.zip', '/content/')

print("unzipped successfully.")

unzipping...
unzipped successfully.


In [ ]:
!pip install tensorflow_model_optimization

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 1.5 MB/s eta 0:00:00


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
import sys
import h5py
import numpy as np
from PIL import Image

# Add the scripts path to the system path
sys.path.append(SCRIPTS_PATH)

# Import the necessary functions from the scripts
from preprocess_data import *
from build_model import *
from train_model import *
from evaluate_model import *

# matplotlib figures appear inline in the notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Auto reload python modules
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [ ]:
# Initialize the TPU
resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
tf.config.experimental_connect_to_cluster(resolver)
tf.tpu.experimental.initialize_tpu_system(resolver)
strategy = tf.distribute.TPUStrategy(resolver)
print("All devices: ", tf.config.list_logical_devices('TPU'))

All devices:  [LogicalDevice(name='/device:TPU:0', device_type='TPU'), LogicalDevice(name='/device:TPU:1', device_type='TPU'), LogicalDevice(name='/device:TPU:2', device_type='TPU'), LogicalDevice(name='/device:TPU:3', device_type='TPU'), LogicalDevice(name='/device:TPU:4', device_type='TPU'), LogicalDevice(name='/device:TPU:5', device_type='TPU'), LogicalDevice(name='/device:TPU:6', device_type='TPU'), LogicalDevice(name='/device:TPU:7', device_type='TPU')]


In [ ]:
"""
  PREPROCESSING THE DATA
  ------------------------
"""
import h5py
import tensorflow as tf

# Paths to HDF5 files
train_file = f'/content/{ZIP_NAME}/train_preprocessed_data.h5'
val_file = f'/content/{ZIP_NAME}/val_preprocessed_data.h5'
test_file = f'/content/{ZIP_NAME}/test_preprocessed_data.h5'

# Dataset parameters
batch_size = 128

with strategy.scope():
    #Create datasets
    train_dn, val_dn, test_dn, train_bm, val_bm, test_bm, class_weight_dn, class_weight_bm = create_datasets(
        train_file, val_file, test_file, batch_size
    )

print("Datasets created successfully.")

Datasets created successfully.


In [ ]:
print("Diagnosis Class Weights:")
print(class_weight_dn)
print("Benign/Malignant Class Weights:")
print(class_weight_bm)

metadata = pd.read_csv(METADATA_PATH)
diagnosis_counts = metadata['diagnosis'].value_counts()
benign_malignant_counts = metadata['benign_malignant'].value_counts()

print("\nDiagnosis Total Distribution:")
print(diagnosis_counts)
print("Benign/Malignant Total Distribution:")
print(benign_malignant_counts)

Diagnosis Class Weights:
{0: 1.1182068400093699, 1: 0.8417421587427538, 2: 1.0896821320550134}
Benign/Malignant Class Weights:
{0: 1.0003405280804694, 1: 0.9996597036804356}

Diagnosis Total Distribution:
diagnosis
nevus       31623
melanoma     7002
other        5408
Name: count, dtype: int64
Benign/Malignant Total Distribution:
benign_malignant
benign       35190
malignant     8843
Name: count, dtype: int64


In [ ]:
"""
  BUILDING THE MODELS
  -----------------
"""
with strategy.scope():
    # Build and compile the ResNet50 models
    diagnosis_model = build_diag_model(learning_rate=1e-3)
    benign_malignant_model = build_bm_model(learning_rate=1e-3)

    # Summarize models
    diagnosis_model.summary()
    benign_malignant_model.summary()

Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_10 (InputLayer)       [(None, 224, 224, 3)]     0         
                                                                 
 resnet50 (Functional)       (None, 7, 7, 2048)        23587712  
                                                                 
 global_average_pooling2d_4  (None, 2048)              0         
  (GlobalAveragePooling2D)                                       
                                                                 
 diagnosis (Dense)           (None, 3)                 6147      
                                                                 
Total params: 23593859 (90.00 MB)
Trainable params: 6147 (24.01 KB)
Non-trainable params: 23587712 (89.98 MB)
_________________________________________________________________
Model: "model_5"
________________________________________________________________

In [ ]:
"""
  TRAINING DIAGNOSIS MODEL
  -------------------
"""
with strategy.scope():
    train_model(
        diagnosis_model,
        train_dn,
        val_dn,
        class_weight_dn,
        epochs=100,
        models_path=MODELS_PATH,
        model_name='diagnosis_model.h5'
    )

    evaluate_model(
        diagnosis_model,
        'diagnosis',
        test_dn,
        results_path=RESULTS_PATH,
        results_file='diagnosis_evaluation_metrics.txt'
    )

Epoch 1/100
597/597 [==============================] - 75s 104ms/step - loss: 1.0774 - accuracy: 0.4109 - val_loss: 1.0034 - val_accuracy: 0.6713 - lr: 0.0010
Epoch 2/100
597/597 [==============================] - 37s 61ms/step - loss: 1.0445 - accuracy: 0.4606 - val_loss: 0.9864 - val_accuracy: 0.6631 - lr: 0.0010
Epoch 3/100
597/597 [==============================] - 37s 62ms/step - loss: 1.0282 - accuracy: 0.4791 - val_loss: 0.9756 - val_accuracy: 0.6625 - lr: 0.0010
Epoch 4/100
597/597 [==============================] - 38s 63ms/step - loss: 1.0176 - accuracy: 0.4908 - val_loss: 0.9678 - val_accuracy: 0.6602 - lr: 0.0010
Epoch 5/100
597/597 [==============================] - 37s 62ms/step - loss: 1.0096 - accuracy: 0.4990 - val_loss: 0.9611 - val_accuracy: 0.6585 - lr: 0.0010
Epoch 6/100
597/597 [==============================] - 37s 61ms/step - loss: 1.0030 - accuracy: 0.5061 - val_loss: 0.9562 - val_accuracy: 0.6568 - lr: 0.0010
Epoch 7/100
597/597 [==============================

In [ ]:
"""
  TRAINING BENIGN/MALIGNANT MODEL
  -------------------
"""
with strategy.scope():
    train_model(
        benign_malignant_model,
        train_bm,
        val_bm,
        class_weight_bm,
        epochs=100,
        models_path=MODELS_PATH,
        model_name='benign_malignant_model.h5'
    )

    evaluate_model(
        benign_malignant_model,
        'benign_malignant',
        test_bm,
        results_path=RESULTS_PATH,
        results_file='benign_malignant_evaluation_metrics.txt'
    )

Epoch 1/100
597/597 [==============================] - ETA: 0s - loss: 0.6756 - accuracy: 0.5820

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


597/597 [==============================] - 73s 99ms/step - loss: 0.6756 - accuracy: 0.5820 - val_loss: 0.6149 - val_accuracy: 0.7769 - lr: 0.0010
Epoch 2/100
597/597 [==============================] - 37s 62ms/step - loss: 0.6579 - accuracy: 0.6187 - val_loss: 0.6029 - val_accuracy: 0.7701 - lr: 0.0010
Epoch 3/100
597/597 [==============================] - 37s 62ms/step - loss: 0.6493 - accuracy: 0.6281 - val_loss: 0.5964 - val_accuracy: 0.7718 - lr: 0.0010
Epoch 4/100
597/597 [==============================] - 37s 61ms/step - loss: 0.6437 - accuracy: 0.6337 - val_loss: 0.5921 - val_accuracy: 0.7732 - lr: 0.0010
Epoch 5/100
597/597 [==============================] - 37s 62ms/step - loss: 0.6396 - accuracy: 0.6379 - val_loss: 0.5889 - val_accuracy: 0.7715 - lr: 0.0010
Epoch 6/100
597/597 [==============================] - 38s 63ms/step - loss: 0.6366 - accuracy: 0.6405 - val_loss: 0.5864 - val_accuracy: 0.7721 - lr: 0.0010
Epoch 7/100
597/597 [==============================] - 37s 62ms/